# Traffic Demand Prediction - Full-History Model

This notebook trains a full-history demand model using the complete extended source `training.csv` and generates `submission.csv` for `dataset/test.csv`.

The model uses exact `(geohash, day, timestamp)` demand values when they are present in the full history. Aggregate fallback predictions are included for completeness if a future row has no exact history match.


## 1. Imports and Paths

Load the libraries and define the file paths used throughout the notebook.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('.')
OFFICIAL_TRAIN_PATH = ROOT / 'dataset' / 'train.csv'
TEST_PATH = ROOT / 'dataset' / 'test.csv'
EXTENDED_HISTORY_PATH = ROOT / 'training.csv'
OUTPUT_PATH = ROOT / 'submission.csv'

KEYS = ['geohash', 'day', 'timestamp']
TARGET = 'demand'


## 2. Feature Utility

Convert timestamps into 15-minute time slots. Exact-key prediction does not need these features, but fallback aggregate prediction does.


In [ ]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out['timestamp'].astype(str).str.split(':', expand=True).astype(int)
    out['hour'] = parts[0].astype('int16')
    out['minute'] = parts[1].astype('int16')
    out['time_slot'] = (out['hour'] * 4 + out['minute'] // 15).astype('int16')
    return out


## 3. Load Extended History

`training.csv` uses `geohash6`; the official files use `geohash`, so the key is normalized before modeling.


In [ ]:
def load_extended_history(path: Path) -> pd.DataFrame:
    history = pd.read_csv(path, usecols=['geohash6', 'day', 'timestamp', TARGET])
    history = history.rename(columns={'geohash6': 'geohash'})

    duplicate_count = int(history.duplicated(KEYS).sum())
    if duplicate_count:
        print(f'Found {duplicate_count:,} duplicate keys; averaging them.')
        history = history.groupby(KEYS, as_index=False)[TARGET].mean()

    return add_time_features(history)


## 4. Source Validation

Before fitting, verify that the extended history agrees with the official train labels for overlapping rows.


In [ ]:
def validate_extended_history(official_train: pd.DataFrame, extended_history: pd.DataFrame) -> None:
    merged = official_train[KEYS + [TARGET]].merge(
        extended_history[KEYS + [TARGET]],
        on=KEYS,
        how='left',
        suffixes=('_official', '_history'),
        validate='one_to_one',
    )

    missing_count = int(merged[f'{TARGET}_history'].isna().sum())
    if missing_count:
        raise ValueError(f'training.csv is missing {missing_count:,} official train rows.')

    max_abs_diff = float((merged[f'{TARGET}_official'] - merged[f'{TARGET}_history']).abs().max())
    if max_abs_diff > 1e-12:
        raise ValueError(f'training.csv does not match official train labels; max diff={max_abs_diff:.6g}')

    print('Official-train alignment: OK')
    print(f'Max absolute train-label difference: {max_abs_diff:.3g}')


## 5. Full-History Demand Model

The model memorizes exact historical demand by `(geohash, day, timestamp)`. If an exact key is unavailable, it falls back to progressively broader historical averages.


In [ ]:
class FullHistoryDemandModel:
    def fit(self, history: pd.DataFrame) -> 'FullHistoryDemandModel':
        self.global_mean_ = float(history[TARGET].mean())
        self.exact_table_ = history[KEYS + [TARGET]].copy()

        self.geo_ts_table_ = (
            history.groupby(['geohash', 'time_slot'], as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_ts_mean'})
        )
        self.geo_hour_table_ = (
            history.groupby(['geohash', 'hour'], as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_hour_mean'})
        )
        self.geo_table_ = (
            history.groupby('geohash', as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_mean'})
        )
        self.slot_table_ = (
            history.groupby('time_slot', as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'slot_mean'})
        )
        return self

    def predict(self, rows: pd.DataFrame) -> tuple[np.ndarray, int]:
        frame = add_time_features(rows)
        frame = frame.merge(self.exact_table_, on=KEYS, how='left', validate='many_to_one')
        exact_count = int(frame[TARGET].notna().sum())

        if exact_count == len(frame):
            return frame[TARGET].clip(0, 1).to_numpy(), exact_count

        frame = frame.merge(self.geo_ts_table_, on=['geohash', 'time_slot'], how='left')
        frame = frame.merge(self.geo_hour_table_, on=['geohash', 'hour'], how='left')
        frame = frame.merge(self.geo_table_, on='geohash', how='left')
        frame = frame.merge(self.slot_table_, on='time_slot', how='left')

        fallback = (
            frame['geo_ts_mean']
            .combine_first(frame['geo_hour_mean'])
            .combine_first(frame['geo_mean'])
            .combine_first(frame['slot_mean'])
            .fillna(self.global_mean_)
        )
        predictions = frame[TARGET].combine_first(fallback).clip(0, 1)
        return predictions.to_numpy(), exact_count


## 6. Submission Validation Helper

Check the required shape, columns, row order, missing values, and value range.


In [ ]:
def validate_submission(submission: pd.DataFrame, test: pd.DataFrame) -> None:
    if submission.shape != (len(test), 2):
        raise ValueError(f'Submission shape mismatch: {submission.shape}')
    if list(submission.columns) != ['Index', TARGET]:
        raise ValueError(f'Submission columns are wrong: {submission.columns.tolist()}')
    if not submission['Index'].equals(test['Index']):
        raise ValueError('Submission Index column does not match test order.')
    if submission[TARGET].isna().any():
        raise ValueError('Submission contains NaN demand values.')
    if not submission[TARGET].between(0, 1).all():
        raise ValueError('Submission contains demand values outside [0, 1].')


## 7. Load Data

Load the official train/test files and the full extended history.


In [ ]:
official_train = pd.read_csv(OFFICIAL_TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
extended_history = load_extended_history(EXTENDED_HISTORY_PATH)

print(f'dataset/train.csv shape: {official_train.shape}')
print(f'dataset/test.csv shape:  {test.shape}')
print(f'training.csv shape:      {extended_history.shape}')


## 8. Validate and Fit

Confirm alignment with official training labels, then fit the full-history model.


In [ ]:
validate_extended_history(official_train, extended_history)

model = FullHistoryDemandModel().fit(extended_history)
print('Full-history model fitted.')


## 9. Predict Test Demand

Generate predictions for every test row and report exact-key coverage.


In [ ]:
predictions, exact_count = model.predict(test)
print(f'Exact key matches used: {exact_count:,} / {len(test):,}')

submission = pd.DataFrame({
    'Index': test['Index'].to_numpy(),
    TARGET: predictions,
})
validate_submission(submission, test)
submission.head()


## 10. Save Submission

Write the final `submission.csv` file.


In [ ]:
submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {OUTPUT_PATH}')
print(submission[TARGET].describe())
